# Hybrid Search with ChromaDB

- ChromaDB Docs: https://docs.trychroma.com/cloud/search-api/hybrid-search
- LangChain Docs: https://docs.langchain.com/oss/python/integrations/vectorstores/chroma
- LangChain + ChromaDB Reference: https://reference.langchain.com/python/langchain-chroma/vectorstores/Chroma/hybrid_search
- Chroma Hybrid Search Using Schema: https://www.trychroma.com/project/sparse-vector-search

Note that the Hybrid Search with Sparse Vector and SearchAPI is **supported in Cloud ChromaDB only**.

## **BM25 Sparse Embeddings**

In [7]:
import chromadb

client = chromadb.PersistentClient(path="vector_store_bm25")

#### **Enable Sparse Vector Embeddings**

To use sparse vectors, add a sparse vector index to your schema. The key parameter is the metadata field name where sparse embeddings will be stored - you can name it whatever you want:

In [8]:
from chromadb import Schema, SparseVectorIndexConfig, K
from chromadb.utils.embedding_functions import ChromaBm25EmbeddingFunction

In [10]:
schema = Schema().create_index(
  key='sparse_vector_key',    
  config=SparseVectorIndexConfig(
    source_key=K.DOCUMENT,
    bm25=True,
    embedding_function=ChromaBm25EmbeddingFunction(
        k=1.2,
        b=0.75,
        avg_doc_length=256.0,
        token_max_length=40
    ),
  )
)

#### **Important**
The source_key specifies which field to generate sparse embeddings from (typically K.DOCUMENT for document text), and embedding_function specifies the function to generate the sparse embeddings. This example uses ChromaCloudSpladeEmbeddingFunction, but you can also use other sparse embedding functions like HuggingFaceSparseEmbeddingFunction or FastembedSparseEmbeddingFunction. The sparse embeddings are automatically generated and stored in the metadata field you specify as the key.

You can customize the BM25 parameters:
- k: Controls term frequency saturation (default: 1.2)
- b: Controls document length normalization (default: 0.75)
- avg_doc_length: Average document length in tokens (default: 256.0)
- token_max_length: Maximum token length (default: 40)
- stopwords: Optional list of stopwords to exclude


In [11]:
collection = client.get_or_create_collection(
  "bm25_collection",
  schema=schema
)

InvalidArgumentError: Failed to reconcile schema: Invalid schema input: Sparse vector indexing is not enabled in local

### **Important**

Sparse vector indexing is only supported on Chroma Cloud, not on local/single-node Chroma. The SparseVectorIndexConfig is disabled by default and requires a Cloud deployment.

In [ ]:
from chromadb import Search, K, Knn, Rrf